# Session 9.1: Model Serialization & Version Control

**Learning Objectives:**
1. Understand production ML workflow
2. Save and load models properly
3. Version control for ML models
4. Create complete prediction pipelines
5. Prepare models for deployment

**Duration:** 2 hours (60 min hands-on)

---

## Part 1: Understanding Model Serialization

**Serialization** means saving your trained model to disk so you can:
- Load it later without retraining
- Deploy it to production
- Share it with others
- Version control it

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import pickle
import joblib
import os
from datetime import datetime

# ML libraries
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, r2_score

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

## Part 2: Saving Scikit-Learn Models

For scikit-learn models, we have two main options:
- **pickle**: Python's built-in serialization
- **joblib**: Better for large numpy arrays (recommended)

In [ ]:
# Create a simple classification dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, 
                          n_redundant=5, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Train a simple model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {accuracy:.4f}")

### Method 1: Using Pickle

In [ ]:
# Create directory for saved models
os.makedirs('saved_models', exist_ok=True)

# Save with pickle
with open('saved_models/rf_classifier_pickle.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model saved with pickle!")

# Load with pickle
with open('saved_models/rf_classifier_pickle.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# Test loaded model
y_pred_loaded = loaded_model.predict(X_test)
accuracy_loaded = accuracy_score(y_test, y_pred_loaded)

print(f"Loaded model accuracy: {accuracy_loaded:.4f}")
print(f"Predictions match: {np.array_equal(y_pred, y_pred_loaded)}")

### Method 2: Using Joblib (Recommended)

In [ ]:
# Save with joblib (more efficient for scikit-learn models)
joblib.dump(model, 'saved_models/rf_classifier_joblib.pkl')
print("Model saved with joblib!")

# Load with joblib
loaded_model_joblib = joblib.load('saved_models/rf_classifier_joblib.pkl')

# Test
y_pred_joblib = loaded_model_joblib.predict(X_test)
print(f"Joblib model accuracy: {accuracy_score(y_test, y_pred_joblib):.4f}")
print(f"Predictions match: {np.array_equal(y_pred, y_pred_joblib)}")

### Comparing File Sizes

In [ ]:
import os

pickle_size = os.path.getsize('saved_models/rf_classifier_pickle.pkl')
joblib_size = os.path.getsize('saved_models/rf_classifier_joblib.pkl')

print(f"Pickle file size: {pickle_size / 1024:.2f} KB")
print(f"Joblib file size: {joblib_size / 1024:.2f} KB")
print(f"\nJoblib is {((pickle_size - joblib_size) / pickle_size * 100):.1f}% smaller (usually)")

## Part 3: Saving Complete Pipelines

**IMPORTANT:** In production, you must save the ENTIRE preprocessing pipeline along with the model!

In [ ]:
# Create a regression dataset
X_reg, y_reg = make_regression(n_samples=1000, n_features=20, n_informative=15, 
                                noise=10, random_state=42)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Create a complete pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train pipeline
pipeline.fit(X_train_reg, y_train_reg)

# Evaluate
y_pred_reg = pipeline.predict(X_test_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
print(f"Pipeline R² score: {r2:.4f}")

In [ ]:
# Save the ENTIRE pipeline
joblib.dump(pipeline, 'saved_models/complete_pipeline.pkl')
print("Complete pipeline saved!")

# Load pipeline
loaded_pipeline = joblib.load('saved_models/complete_pipeline.pkl')

# Test on new data (no preprocessing needed!)
y_pred_loaded_pipeline = loaded_pipeline.predict(X_test_reg)

print(f"Loaded pipeline R²: {r2_score(y_test_reg, y_pred_loaded_pipeline):.4f}")
print(f"Predictions match: {np.allclose(y_pred_reg, y_pred_loaded_pipeline)}")

## Part 4: Saving Keras/TensorFlow Models

For deep learning models, use TensorFlow's built-in save methods.

In [ ]:
# Create a simple neural network
def create_model(input_dim):
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam',
                 loss='binary_crossentropy',
                 metrics=['accuracy'])
    return model

# Create and train model
nn_model = create_model(X_train.shape[1])
print("Neural Network Architecture:")
nn_model.summary()

In [ ]:
# Train model
history = nn_model.fit(X_train, y_train, 
                       epochs=20, 
                       batch_size=32,
                       validation_split=0.2,
                       verbose=0)

# Evaluate
test_loss, test_acc = nn_model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

### Method 1: SavedModel Format (Recommended)

In [ ]:
# Save in SavedModel format (recommended)
nn_model.save('saved_models/nn_classifier_savedmodel')
print("Model saved in SavedModel format!")

# Load SavedModel
loaded_nn = keras.models.load_model('saved_models/nn_classifier_savedmodel')

# Test
loaded_loss, loaded_acc = loaded_nn.evaluate(X_test, y_test, verbose=0)
print(f"Loaded model accuracy: {loaded_acc:.4f}")

### Method 2: HDF5 Format (Legacy but common)

In [ ]:
# Save in HDF5 format
nn_model.save('saved_models/nn_classifier.h5')
print("Model saved in HDF5 format!")

# Load HDF5
loaded_nn_h5 = keras.models.load_model('saved_models/nn_classifier.h5')

# Test
h5_loss, h5_acc = loaded_nn_h5.evaluate(X_test, y_test, verbose=0)
print(f"HDF5 model accuracy: {h5_acc:.4f}")

## Part 5: Model Versioning

Best practice: Use semantic versioning (MAJOR.MINOR.PATCH)

In [ ]:
class ModelVersion:
    """Model versioning utility"""
    
    def __init__(self, major=1, minor=0, patch=0):
        self.major = major
        self.minor = minor
        self.patch = patch
        
    def __str__(self):
        return f"v{self.major}.{self.minor}.{self.patch}"
    
    def increment_major(self):
        """Breaking changes, complete retrain"""
        self.major += 1
        self.minor = 0
        self.patch = 0
        
    def increment_minor(self):
        """New features, significant improvements"""
        self.minor += 1
        self.patch = 0
        
    def increment_patch(self):
        """Bug fixes, minor improvements"""
        self.patch += 1

# Example usage
version = ModelVersion(1, 0, 0)
print(f"Initial version: {version}")

version.increment_minor()
print(f"After adding features: {version}")

version.increment_patch()
print(f"After bug fix: {version}")

In [ ]:
def save_model_with_metadata(model, model_name, version, metrics, description=""):
    """
    Save model with version and metadata
    
    Args:
        model: Trained model
        model_name: Name of the model
        version: ModelVersion object
        metrics: Dictionary of performance metrics
        description: Model description
    """
    # Create version directory
    version_dir = f'saved_models/{model_name}/{version}'
    os.makedirs(version_dir, exist_ok=True)
    
    # Save model
    model_path = f'{version_dir}/model.pkl'
    joblib.dump(model, model_path)
    
    # Save metadata
    metadata = {
        'version': str(version),
        'created_at': datetime.now().isoformat(),
        'metrics': metrics,
        'description': description,
        'model_type': type(model).__name__
    }
    
    import json
    with open(f'{version_dir}/metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"Model saved: {model_path}")
    print(f"Metadata saved: {version_dir}/metadata.json")
    return version_dir

# Save model with versioning
version = ModelVersion(1, 0, 0)
metrics = {'accuracy': accuracy, 'test_samples': len(y_test)}

save_model_with_metadata(
    model=model,
    model_name='spam_classifier',
    version=version,
    metrics=metrics,
    description='Random Forest classifier for spam detection'
)

In [ ]:
def load_model_with_metadata(model_name, version):
    """
    Load model with metadata
    
    Returns:
        tuple: (model, metadata)
    """
    version_dir = f'saved_models/{model_name}/{version}'
    
    # Load model
    model = joblib.load(f'{version_dir}/model.pkl')
    
    # Load metadata
    import json
    with open(f'{version_dir}/metadata.json', 'r') as f:
        metadata = json.load(f)
    
    return model, metadata

# Load versioned model
loaded_model, metadata = load_model_with_metadata('spam_classifier', 'v1.0.0')

print("Model loaded successfully!")
print(f"\nMetadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")

## Part 6: Creating a Model Card

A **model card** documents your model's purpose, performance, and limitations.

In [ ]:
def create_model_card(model_name, version, model_info):
    """
    Create a model card (README) for the model
    
    Args:
        model_name: Name of model
        version: Version string
        model_info: Dictionary with model information
    """
    card = f"""# {model_name} - {version}

## Model Overview
{model_info.get('description', 'No description provided')}

## Model Details
- **Model Type:** {model_info.get('model_type', 'Unknown')}
- **Version:** {version}
- **Created:** {model_info.get('created_at', 'Unknown')}
- **Framework:** {model_info.get('framework', 'scikit-learn')}

## Performance
### Training Metrics
"""    
    
    if 'metrics' in model_info:
        for metric, value in model_info['metrics'].items():
            card += f"- **{metric}:** {value}\n"
    
    card += f"""
## Intended Use
{model_info.get('intended_use', 'General purpose classification/regression')}

## Limitations
{model_info.get('limitations', 'Model may not generalize to data outside training distribution')}

## How to Use
```python
import joblib

# Load model
model = joblib.load('model.pkl')

# Make predictions
predictions = model.predict(X_new)
```

## Training Data
{model_info.get('training_data', 'See documentation for details')}

## License
{model_info.get('license', 'MIT')}
"""
    
    return card

# Create model card
model_info = {
    'description': 'Random Forest classifier for spam detection',
    'model_type': 'RandomForestClassifier',
    'created_at': datetime.now().strftime('%Y-%m-%d'),
    'framework': 'scikit-learn',
    'metrics': {'accuracy': 0.95, 'f1_score': 0.94},
    'intended_use': 'Detecting spam in email messages',
    'limitations': 'May not work well on non-English text or very short messages',
    'training_data': 'Email dataset with 10,000 messages',
    'license': 'MIT'
}

card = create_model_card('Spam Classifier', 'v1.0.0', model_info)

# Save model card
with open('saved_models/spam_classifier/v1.0.0/MODEL_CARD.md', 'w') as f:
    f.write(card)

print("Model card created!")
print("\n" + "="*50)
print(card)
print("="*50)

## Part 7: Error Handling in Production

Always handle errors gracefully!

In [ ]:
def safe_load_model(model_path):
    """
    Safely load a model with error handling
    
    Args:
        model_path: Path to model file
        
    Returns:
        model or None if error
    """
    try:
        # Check if file exists
        if not os.path.exists(model_path):
            print(f"Error: Model file not found at {model_path}")
            return None
        
        # Try loading
        model = joblib.load(model_path)
        print(f"Model loaded successfully from {model_path}")
        return model
        
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        return None

def safe_predict(model, X):
    """
    Safely make predictions with error handling
    
    Args:
        model: Trained model
        X: Input features
        
    Returns:
        predictions or None if error
    """
    try:
        # Validate input
        if X is None or len(X) == 0:
            print("Error: Empty input")
            return None
        
        # Make prediction
        predictions = model.predict(X)
        return predictions
        
    except Exception as e:
        print(f"Error making prediction: {str(e)}")
        return None

# Test error handling
print("Testing error handling...\n")

# Test 1: Valid model
model = safe_load_model('saved_models/rf_classifier_joblib.pkl')

# Test 2: Invalid path
model = safe_load_model('nonexistent_model.pkl')

# Test 3: Valid prediction
model = joblib.load('saved_models/rf_classifier_joblib.pkl')
predictions = safe_predict(model, X_test[:5])
print(f"Predictions: {predictions}")

## Part 8: Complete Model Manager Class

Let's create a production-ready model manager.

In [ ]:
# This code will be saved as model_version_manager.py

class ModelManager:
    """
    Complete model management system with versioning, 
    metadata, and error handling.
    """
    
    def __init__(self, base_dir='saved_models'):
        self.base_dir = base_dir
        os.makedirs(base_dir, exist_ok=True)
    
    def save(self, model, name, version, metrics, description=""):
        """
        Save model with version and metadata
        """
        # Create directory structure
        model_dir = os.path.join(self.base_dir, name, version)
        os.makedirs(model_dir, exist_ok=True)
        
        # Save model
        model_path = os.path.join(model_dir, 'model.pkl')
        joblib.dump(model, model_path)
        
        # Save metadata
        metadata = {
            'name': name,
            'version': version,
            'created_at': datetime.now().isoformat(),
            'metrics': metrics,
            'description': description,
            'model_type': type(model).__name__
        }
        
        import json
        metadata_path = os.path.join(model_dir, 'metadata.json')
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✓ Model saved: {name} {version}")
        print(f"  Location: {model_path}")
        return model_dir
    
    def load(self, name, version='latest'):
        """
        Load model by name and version
        """
        try:
            if version == 'latest':
                # Find latest version
                model_base = os.path.join(self.base_dir, name)
                if not os.path.exists(model_base):
                    raise FileNotFoundError(f"No model found: {name}")
                
                versions = [d for d in os.listdir(model_base) 
                           if os.path.isdir(os.path.join(model_base, d))]
                version = sorted(versions)[-1]  # Get latest
            
            # Load model
            model_path = os.path.join(self.base_dir, name, version, 'model.pkl')
            model = joblib.load(model_path)
            
            # Load metadata
            import json
            metadata_path = os.path.join(self.base_dir, name, version, 'metadata.json')
            with open(metadata_path, 'r') as f:
                metadata = json.load(f)
            
            print(f"✓ Model loaded: {name} {version}")
            return model, metadata
            
        except Exception as e:
            print(f"✗ Error loading model: {str(e)}")
            return None, None
    
    def list_models(self):
        """
        List all saved models
        """
        models = []
        
        if not os.path.exists(self.base_dir):
            return models
        
        for model_name in os.listdir(self.base_dir):
            model_path = os.path.join(self.base_dir, model_name)
            if os.path.isdir(model_path):
                versions = [d for d in os.listdir(model_path) 
                           if os.path.isdir(os.path.join(model_path, d))]
                models.append({
                    'name': model_name,
                    'versions': sorted(versions)
                })
        
        return models
    
    def compare_versions(self, name, version1, version2):
        """
        Compare two model versions
        """
        import json
        
        # Load metadata for both versions
        meta1_path = os.path.join(self.base_dir, name, version1, 'metadata.json')
        meta2_path = os.path.join(self.base_dir, name, version2, 'metadata.json')
        
        with open(meta1_path, 'r') as f:
            meta1 = json.load(f)
        
        with open(meta2_path, 'r') as f:
            meta2 = json.load(f)
        
        print(f"\nComparing {name}: {version1} vs {version2}")
        print("="*50)
        
        # Compare metrics
        print("\nMetrics Comparison:")
        all_metrics = set(meta1.get('metrics', {}).keys()) | set(meta2.get('metrics', {}).keys())
        
        for metric in all_metrics:
            val1 = meta1.get('metrics', {}).get(metric, 'N/A')
            val2 = meta2.get('metrics', {}).get(metric, 'N/A')
            
            if val1 != 'N/A' and val2 != 'N/A':
                diff = val2 - val1
                symbol = '↑' if diff > 0 else '↓' if diff < 0 else '→'
                print(f"  {metric}: {val1:.4f} → {val2:.4f} {symbol}")
            else:
                print(f"  {metric}: {val1} → {val2}")

# Test the ModelManager
manager = ModelManager()

# Save a model
manager.save(
    model=model,
    name='demo_classifier',
    version='v1.0.0',
    metrics={'accuracy': 0.95, 'precision': 0.94},
    description='Demo random forest classifier'
)

# List all models
print("\nAll saved models:")
all_models = manager.list_models()
for m in all_models:
    print(f"  {m['name']}: {', '.join(m['versions'])}")

# Load latest version
loaded_model, metadata = manager.load('demo_classifier', 'latest')
print(f"\nLoaded metadata: {metadata}")

## Summary & Key Takeaways

### What We Learned:
1. **Model Serialization:**
   - Use `joblib` for scikit-learn models
   - Use `model.save()` for Keras/TensorFlow
   - Always save the ENTIRE pipeline (preprocessing + model)

2. **Version Control:**
   - Use semantic versioning (MAJOR.MINOR.PATCH)
   - Save metadata with each version
   - Create model cards for documentation

3. **Production Best Practices:**
   - Error handling is critical
   - Validate inputs before prediction
   - Log everything for debugging

### Next Session:
We'll build REST APIs with Flask and FastAPI to serve these models!

---

**Assignment:**
1. Save one of your models from previous modules with proper versioning
2. Create a model card documenting its performance
3. Test loading and making predictions

**Estimated Time:** 30-45 minutes